Graphing Simulated Chaotic Motion

In [5]:
import socket
import struct
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def recieve_doubles(conn):
    # Step 1: Read the 8-byte size_t header
    len_bytes = conn.recv(8)
    if len(len_bytes) < 8:
        raise ConnectionError("Incomplete size header received")

    total_bytes_to_receive = struct.unpack('Q', len_bytes)[0]

    # Step 2: Handle empty payload
    if total_bytes_to_receive == 0:
        return np.array([], dtype=np.float64)

    # Step 3: Read the payload (raw binary double array)
    data = bytearray()
    while len(data) < total_bytes_to_receive:
        packet = conn.recv(total_bytes_to_receive - len(data))
        if not packet:
            raise ConnectionError("Socket connection closed before receiving all data")
        data.extend(packet)

    # Step 4: Convert to NumPy float64 array
    return np.frombuffer(data, dtype=np.float64)

In [7]:
def server_program():
    host = '127.0.0.1'  # localhost
    port = 8080
    
    # get instance
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_socket.bind((host, port))
    
    # set number of clients server can listen to simultaneously
    server_socket.listen(1)
    conn, address = server_socket.accept()
    print(f"Connection from {address}")
    
    arr = recieve_doubles(conn)
    print("Sending confirmation...")
    conn.sendall(b"OK")  # confirm all data recieved
    
    conn.close()  # close the connection
    server_socket.close()
    print("Server connection closed.")
    
    return arr

In [8]:
if __name__ == '__main__':
    raw_arr = server_program()

Connection from ('127.0.0.1', 38684)
Sending confirmation...
Server connection closed.


In [9]:
print(raw_arr)

[0. 1. 2. 3. 4. 5. 6. 7.]


In [ ]:
test_arr1 = np.array(raw_arr[::2])
test_arr2 = 
np.column_stack((raw_arr[::2], raw_arr[1::2]))